In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

GROQ_BASE_URL = "https://api.groq.com/openai/v1"

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")
    
MODEL = "openai/gpt-oss-120b"
openai = OpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)


GROQ API Key exists and begins gsk_4D0P


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print (f"Response: {response.choices[0].finish_reason}")

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Tool calls: {message.tool_calls}")
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [6]:
import sqlite3

In [15]:
DB = "airline.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    cursor.execute('CREATE TABLE IF NOT EXISTS weather (city TEXT PRIMARY KEY, temperature REAL, description TEXT, FOREIGN KEY(city) REFERENCES prices(city))')
    # cursor.execute('CREATE TABLE IF NOT EXISTS flights (flight_number TEXT PRIMARY KEY, departure_city TEXT, arrival_city TEXT, departure_time TEXT, arrival_time TEXT)')
    conn.commit()

In [8]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [9]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [10]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [11]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [12]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [16]:
weather_info = {
    "london": {
        "temperature": "18–25°C",
        "description": "Cool and mild with frequent rainfall throughout the year."
    },
    "paris": {
        "temperature": "20–30°C",
        "description": "Mild climate with warm summers and cool winters."
    },
    "tokyo": {
        "temperature": "28–35°C",
        "description": "Hot and humid summers with cool winters."
    },
    "sydney": {
        "temperature": "18–30°C",
        "description": "Mild to warm climate with seasons opposite to the Northern Hemisphere."
    }
}

In [17]:
def set_weather_info(city, temperature, description):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO weather (city, temperature, description) VALUES (?, ?, ?) ON CONFLICT(city) DO UPDATE SET temperature = ?, description = ?', (city.lower(), temperature, description, temperature, description))
        conn.commit()

In [18]:
for city, info in weather_info.items():
    set_weather_info(city, info["temperature"], info["description"])    

In [20]:
def get_weather_info(city):
    print(f"DATABASE TOOL CALLED: Getting weather info for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT temperature, description FROM weather WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        if result:
            temperature, description = result
            return f"Weather in {city}: {temperature}, {description}"
        else:
            return "No weather data available for this city"

In [21]:
get_weather_info("London")

DATABASE TOOL CALLED: Getting weather info for London


'Weather in London: 18–25°C, Cool and mild with frequent rainfall throughout the year.'

In [22]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:

weather_function = {
    "name": "get_weather_info",
    "description": "Get the weather information for a city user asked to book flight to. Give the temperature and a short description of the weather.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [25]:
tools = [{"type": "function", "function": price_function}, {"type": "function", "function": weather_function}]

In [26]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print (f"Response: {response.choices[0].finish_reason}")

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Tool calls: {message.tool_calls}")
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [28]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_weather_info":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            weather_details = get_weather_info(city)
            responses.append({
                "role": "tool",
                "content": weather_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Response: stop
Response: stop
Response: tool_calls
Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_538915ed-06fa-4bab-a532-0fa8e39c1e89', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')]
DATABASE TOOL CALLED: Getting price for Tokyo
Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_42283cdd-bd4c-44f4-ba33-f344c2df246e', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_weather_info'), type='function')]
DATABASE TOOL CALLED: Getting weather info for Tokyo
